In [2]:
import torch
import torch.nn as nn

class RiskWorldModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2):
        super(RiskWorldModel, self).__init__()
        # input_dim = DNA_features + Environment_features
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2), # Prevent overfitting on training eras
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x shape: [batch_size, sequence_length, input_dim]
        out, _ = self.gru(x)
        
        # We only need the risk prediction at the final horizon T
        final_timestep_out = out[:, -1, :]
        risk_prob = self.fc(final_timestep_out)
        
        return risk_prob